# Supplementary tables that were compiled by hand

Supplementary Tables 3, 8, 10 and 11 were **assembled manually**, not emitted by any script, so
they are not rebuilt by `01_supplementary_tables.ipynb`. The published sheets are kept verbatim in
`manual/`, and this notebook recomputes the numbers inside them so they can be refreshed if the
upstream data changes.

**It does not write the spreadsheets.** It writes one CSV per subtable to `manual/refreshed/`, next
to the published value, so the numbers can be checked and pasted in deliberately. Nothing here
overwrites a published sheet.

| Table | Handled here | Why |
| --- | --- | --- |
| ST8, subgroup analysis | yes, all four subtables | every number is a count over the PAV-supported associations |
| ST11, colocalisation overlap | **no** | static asset in `assets/`; hand-made, definitions not recoverable |
| ST3, GSEA | **no** | static asset in `assets/`; written by hand by Polina |
| ST10, fine-mapping statistics | not yet | compiled from numbers spread across several notebooks |

In [1]:
import pandas as pd
import pyarrow.dataset as pads
from pyspark.sql import functions as f

from gentropy.common.session import Session
from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

out_dir = paper.ROOT / "chapters" / "06-supplementary-tables" / "manual" / "refreshed"
out_dir.mkdir(parents=True, exist_ok=True)

CLPP, H4 = 0.01, 0.8  # the significance thresholds used throughout the pipeline
print(out_dir)

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 22:51:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


/Users/yt4/Projects/Gentropy-manuscript/chapters/06-supplementary-tables/manual/refreshed


## Supplementary Table 8 — subgroup analysis

Four subtables. The first two cover every nominated gene-disease association; the last two the
PAV-supported subgroup that the manuscript singles out:

1. all nominated genetic evidence by ChEMBL level-1 target class
2. the same by therapeutic area
3. the "best subgroup" (PAV **and** 2-5 therapeutic areas) by target class
4. the same by therapeutic area

`Number of Pairs (N)` counts distinct gene-disease pairs, not credible sets.

In [2]:
# ChEMBL level-1 target class. Genes with no ChEMBL class are dropped, not bucketed into
# "Unclassified protein" — that is itself a ChEMBL class, and bucketing inflates it about 14-fold.
target = (
    pads.dataset(str(paper.RELEASE / "target"), format="parquet").to_table(columns=["id", "targetClass"]).to_pandas()
)


def level_1_class(classes):
    """The ChEMBL level-1 label, or None when the target carries no class."""
    if classes is None or len(classes) == 0:
        return None
    for entry in classes:
        if entry["level"] == "l1":
            return entry["label"]
    return None


target["l1_class"] = target["targetClass"].apply(level_1_class)
target_class = target.dropna(subset=["l1_class"]).set_index("id")["l1_class"]
print(f"targets with a ChEMBL level-1 class: {len(target_class):,}")

disease_area = pads.dataset(str(paper.derived("efo_therapeutic_area")), format="parquet").to_table().to_pandas()
area_of = disease_area.set_index("id")["primaryTherapeuticArea"]
area_name = {v: k for k, v in paper.THERAPEUTIC_AREAS.items()}
print(f"diseases with a therapeutic area: {len(area_of):,}")

targets with a ChEMBL level-1 class: 4,928
diseases with a therapeutic area: 12,856


In [3]:
def pair_frame(rows):
    """Distinct gene-disease pairs from a table carrying `geneId` and a `diseaseIds` list."""
    pairs = rows.explode("diseaseIds")[["geneId", "diseaseIds"]].dropna().drop_duplicates()
    return pairs.rename(columns={"diseaseIds": "diseaseId"})


def by_target_class(pairs):
    """ST8 Tables 1 and 3: pair, disease and target counts per ChEMBL level-1 class."""
    p = pairs.assign(l1_class=pairs["geneId"].map(target_class)).dropna(subset=["l1_class"])
    out = p.groupby("l1_class").agg(
        **{
            "Number of Pairs (N)": ("geneId", "size"),
            "Unique Diseases": ("diseaseId", "nunique"),
            "Unique Targets": ("geneId", "nunique"),
        }
    )
    return out.rename_axis("Target Class (l1_class)").reset_index()


def by_therapeutic_area(pairs):
    """ST8 Tables 2 and 4: the same counts per therapeutic area of the disease."""
    p = pairs.assign(area=pairs["diseaseId"].map(area_of)).dropna(subset=["area"])
    out = p.groupby("area").agg(
        **{
            "Number of Pairs (N)": ("geneId", "size"),
            "Unique Diseases": ("diseaseId", "nunique"),
            "Unique Targets": ("geneId", "nunique"),
        }
    )
    out = out.rename_axis("Ontology ID").reset_index()
    out.insert(0, "Therapeutic Area", out["Ontology ID"].map(area_name).fillna("other (no area root)"))
    return out.sort_values("Number of Pairs (N)", ascending=False)

In [4]:
# Subtables 1 and 2 cover "all nominated genetic evidence" — every L2G-prioritised gene-disease
# association, with no PAV filter. Filtering to VEP == 1 here undercounts by a factor of about six.
nominated = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases")).select("geneId", "diseaseIds").toPandas()
)
pairs_all = pair_frame(nominated)
print(f"all nominated gene-disease pairs: {len(pairs_all):,}")

# Subtables 3 and 4: the published "best subgroup" — PAV and 2 to 5 therapeutic areas.
# ST7 is already one row per credible set and disease, so no list parsing is needed. Do not
# `ast.literal_eval` a numpy-repr column: "['A' 'B' 'C']" parses as implicit string concatenation
# into the single token 'ABC', which silently collapses every multi-disease credible set.
best = pd.read_csv(paper.ROOT / "chapters/06-supplementary-tables/sheets/ST7_pav_gene_disease_pairs.csv")
pairs_best = best[["geneId", "diseaseId"]].dropna().drop_duplicates()
print(f"PAV + 2-5 therapeutic areas pairs:    {len(pairs_best):,}")

all nominated gene-disease pairs: 36,858
PAV + 2-5 therapeutic areas pairs:    2,734


In [5]:
subtables = {
    "ST8_table1_all_nominated_by_target_class": by_target_class(pairs_all),
    "ST8_table2_all_nominated_by_therapeutic_area": by_therapeutic_area(pairs_all),
    "ST8_table3_best_subgroup_by_target_class": by_target_class(pairs_best),
    "ST8_table4_best_subgroup_by_therapeutic_area": by_therapeutic_area(pairs_best),
}
for name, table in subtables.items():
    table.to_csv(out_dir / f"{name}.csv", index=False)
    print(f"{name}: {len(table)} rows, {int(table['Number of Pairs (N)'].sum()):,} pairs")
subtables["ST8_table3_best_subgroup_by_target_class"]

ST8_table1_all_nominated_by_target_class: 15 rows, 13,112 pairs
ST8_table2_all_nominated_by_therapeutic_area: 23 rows, 36,858 pairs
ST8_table3_best_subgroup_by_target_class: 13 rows, 965 pairs
ST8_table4_best_subgroup_by_therapeutic_area: 23 rows, 2,734 pairs


,Target Class (l1_class),Number of Pairs (N),Unique Diseases,Unique Targets
0,Adhesion,25,21,7
1,Enzyme,412,220,146
2,Epigenetic regulator,23,20,8
3,Ion channel,42,33,15
4,Membrane receptor,136,93,37
5,Other cytosolic protein,10,10,6
6,Other membrane protein,1,1,1
7,Other nuclear protein,5,5,2
8,Secreted protein,54,46,13
9,Structural protein,47,40,16


In [6]:
# Against the published sheet, so any drift is visible rather than assumed away.
published_table3 = {
    "Enzyme": 441,
    "Other nuclear protein": 5,
    "Adhesion": 25,
    "Surface antigen": 0,
    "Membrane receptor": 136,
    "Epigenetic regulator": 23,
    "Other cytosolic protein": 10,
    "Other membrane protein": 1,
    "Ion channel": 42,
    "Auxiliary transport protein": 0,
    "Transcription factor": 23,
    "Unclassified protein": 125,
    "Transporter": 67,
    "Secreted protein": 56,
    "Structural protein": 47,
}
check = subtables["ST8_table3_best_subgroup_by_target_class"].set_index("Target Class (l1_class)")
comparison = (
    pd.DataFrame({"published": pd.Series(published_table3), "recomputed": check["Number of Pairs (N)"]})
    .fillna(0)
    .astype(int)
)
comparison["difference"] = comparison["recomputed"] - comparison["published"]
comparison.to_csv(out_dir / "ST8_table3_vs_published.csv")
print(f"published total {comparison['published'].sum():,} | recomputed {comparison['recomputed'].sum():,}")
comparison

published total 1,001 | recomputed 965


,published,recomputed,difference
Adhesion,25,25,0
Auxiliary transport protein,0,0,0
Enzyme,441,412,-29
Epigenetic regulator,23,23,0
Ion channel,42,42,0
Membrane receptor,136,136,0
Other cytosolic protein,10,10,0
Other membrane protein,1,1,0
Other nuclear protein,5,5,0
Secreted protein,56,54,-2


### Reading the ST8 comparison

The class structure reproduces exactly — every category is present with the right shape — but the
recomputed counts run uniformly a little low. That deficit is inherited, not introduced here:
`ST7_pav_gene_disease_pairs.csv` holds 4,316 rows against the published sheet's 4,742, and Tables 3
and 4 are a breakdown of exactly that set. **Fix ST7 and these follow.** Do not tune the numbers
here to close the gap.

## Supplementary Table 11 — not recomputed

ST11 is a **static asset**: `assets/ST11_-_coloc_overlap.xlsx`, copied from the manuscript tree and
shipped as-is. It was compiled by hand and its definitions could not be recovered — a recomputation
of both subtables disagreed with the published values in both directions, so nothing here derives
it. Treat the sheet as the source of truth.

## Supplementary Table 3 — not recomputed

ST3 is a **static asset**: `assets/ST3_-_GSEA_results.xlsx`, copied from the manuscript tree and
shipped as-is. It was written by hand by Polina and is not rebuilt here.

Recorded only so the provenance is not lost: the gene sets are public, and the sheet's own `Source`
column names the exact Enrichr libraries — `Reactome_Pathways_2024` (2,105 sets) and `KEGG_2026`
(352), whose sizes sum to 2,457, exactly the published row count. The enrichment was run with
`blitzgsea`, which is stochastic. The Results section 5 pathway counts (312 / 221 / 91) come from
this same manual analysis.